# CellNeighborEX v2 Tutorial — 04. Identifying Tumor Subtype-Specific CCI Genes in Human Ovarian Cancer (OVC)

**Key concept:** In this tutiroal, ovarian cancer exhibits distinct tumor subtypes, providing a useful setting for examining how CCI genes and cell–cell interaction signals vary across tumor subtype–defined regions.

This notebook starts from bundled **precomputed `ccisignal` outputs**. We therefore skip the deconvolution step and focus on running `ccigenes` and `ccipairs` modules to identify tumor subtype–specific CCI genes and their interacting cell type pairs.

## Core Concept: Comparing tumor subtype CCI signals in Ovarian cancer

**Human ovarian cancer has multiple tumor subtypes. Here, tumor subtypes are defined based on copy number alteration profiles. This dataset therefore provides a useful example for investigating how tumor subtype definitions influence the identification of tumor subtype-specific CCI genes and their interacting cell type pairs.**

### Using human ovarian cancer spatial transcriptomics data as an example, this tutorial demonstrates how to:

- Analyze spatial cell-type distributions across tumor subtype–defined regions
- Identify tumor subtype–specific CCI genes 
- Infer interacting cell type pairs for each CCI gene

## ⚙️ User Setup

Before running this notebook, set `RUN_CODE_DIR` in the cell below to the path of your local `CNEXv2/run_code` directory.

**Example:**
- Linux / Mac: `Path("/home/username/CNEXv2/run_code")`
- Windows: `Path("C:/Users/username/CNEXv2/run_code")`


In [ ]:
from pathlib import Path

# ============================================================
#  USER CONFIGURATION — Edit before running
# ============================================================
RUN_CODE_DIR = Path("/path/to/CNEXv2/run_code").resolve()  # ← Change to your path
# ============================================================


In [ ]:
import os
import sys

# --- Environment Configuration ---
# Set environment variables to control multi-threading for linear algebra libraries.
# This ensures computational stability and prevents CPU over-subscription.
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

# --- Standard Library and Data Processing Imports ---
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# --- Path Management ---
BASE_DIR = RUN_CODE_DIR.parent  # CNEXv2

# Add the base directory to the system path to enable imports from the local CellNeighborEX package.
sys.path.insert(0, str(BASE_DIR))

# --- CellNeighborEX Module Imports ---

# 1. ccisignal: Functions for calculating cell type proportions and spatial clustering.
from CellNeighborEX.ccisignal import compute_proportion, cluster_spots_by_proportion

# 2. ccigenes: Functions for identifying and statistically validating CCI-associated genes.
from CellNeighborEX.ccigenes import (
    clean_column_names,
    obtain_clean_celltype_names,
    permutation_test_all_clusters,
    chi_square_goodness_of_fit,
    compute_combined_p_value,
    simplify_gene_names,
)

# 3. ccipairs: Functions for modeling cell-type interaction terms and database validation.
from CellNeighborEX.ccipairs import (
    regress_residual_on_interaction_with_regularization,
    extract_interaction_terms,
    compare_database,
)

In [ ]:
# --- Data Directory Configuration ---
# Define the path for the ovarian cancer tutorial dataset
DATA_DIR = BASE_DIR / "Tutorial-ExampleData" / "4_ovarian_cancer"
# File path for single-cell RNA-seq reference data
SC_REF_FILE = DATA_DIR / "sc_ccisignal.h5ad"
# File path for spatial Visium transcriptomics data
VISIUM_FILE = DATA_DIR / "sp_ccisignal.h5ad"

# --- Metadata and Organism Settings ---
# Set species to 'human' for ligand-receptor database mapping
SPECIES = "human"

# --- Spatial Clustering Configuration ---
# 'cluster' is the default annotation for this ovarian cancer dataset.
# Alternatively, 'proportion_leiden' can be used for niche-based analysis.
CLUSTER_INFO = "cluster"  # Options: 'cluster', 'proportion_leiden'

# --- Analysis Hyperparameters (Speed Knobs) ---
# Number of iterations for the permutation test
N_PERMUTATIONS = 250
# Threshold for Log Fold Change filtering of CCI genes
LOG_FC = 0.5
# P-value threshold for identifying significant interaction terms in the regression model
PVAL_TERM = 0.05

# --- Output and Figure Directory Management ---
# Create the primary output directory for this analysis run
OUTPUT_DIR = RUN_CODE_DIR / "tutorial_output" / "04_ovarian_cancer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create a dedicated subdirectory for plot exports
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# --- Scanpy Plotting Environment ---
# Set the global save directory for Scanpy-generated figures
sc.settings.figdir = str((OUTPUT_DIR / "figures").resolve())
# Configure figure resolution (DPI) for high-quality visualization
sc.settings.set_figure_params(dpi=120)

# --- Execution Log ---
# Print the configured paths and settings to the console for verification
print(f"SC_REF_FILE : {SC_REF_FILE}")
print(f"VISIUM_FILE : {VISIUM_FILE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"CLUSTER_INFO: {CLUSTER_INFO}")

## 1) Load data and check metadata information

The following datasets are loaded:
- `sc_ccisignal.h5ad`: reference signatures
- `sp_ccisignal.h5ad`: spatial deconvolution map

In [ ]:
# --- Data Loading ---
# Load the single-cell reference signature and the spatial Visium data
adata_ref_sig = sc.read_h5ad(SC_REF_FILE)
adata_vis = sc.read_h5ad(VISIUM_FILE)

# Display high-level summaries of the loaded objects
print("Reference:", adata_ref_sig)
print("Spatial:  ", adata_vis)

# --- Model Integrity Check ---
# Ensure that both datasets contain the 'mod' key in .uns, 
# which stores critical deconvolution and model parameters
assert "mod" in adata_ref_sig.uns and "mod" in adata_vis.uns

# --- Metadata Preprocessing ---
# If a 'sample' identifier is missing, extract the first available sample name 
# from the spatial metadata dictionary
if "sample" not in adata_vis.obs.columns and "spatial" in adata_vis.uns:
    adata_vis.obs["sample"] = list(adata_vis.uns["spatial"].keys())[0]

# --- Cluster Column Verification ---
# Identify which clustering or niche annotation columns are available in the spatial data
available = [c for c in ["cluster", "proportion_leiden", "leiden"] if c in adata_vis.obs.columns]
print("Cluster columns present:", available)

# Preview the metadata table, prioritizing the identified cluster columns
display(adata_vis.obs[available].head() if available else adata_vis.obs.head())

## 2) Build observed/expected expression matrices

Upon completing the `ccisignal` module, you typically obtain two primary output files: `sc_ccisignal.h5ad` and `sp_ccisignal.h5ad`. For this tutorial, these two files are provided as precomputed outputs to allow you to proceed directly with the analysis. Using these datasets, we calculate the expected expression values to compare them against the observed expression data.

In [ ]:
# --- Setup and Identifier Simplification ---
cluster_info = CLUSTER_INFO

# Normalize gene names to a standard format (e.g., handling case sensitivity or prefix removal)
simplify_gene_names(adata_ref_sig, SPECIES)
simplify_gene_names(adata_vis, SPECIES)

# --- Reference Signature Extraction (Gene × Cell Type) ---
# Retrieve the list of cell type factors from the reference model
factor_names = list(adata_ref_sig.uns["mod"]["factor_names"])
# Extract the mean expression per cluster; handle different data structures (DataFrame vs. Array)
inf_raw = adata_ref_sig.varm.get("means_per_cluster_mu_fg", None)

if inf_raw is None:
    inf_aver2 = adata_ref_sig.obs[factor_names].copy()
elif isinstance(inf_raw, pd.DataFrame):
    inf_aver2 = inf_raw.copy()
else:
    inf_aver2 = pd.DataFrame(inf_raw, index=adata_ref_sig.var_names, columns=factor_names)

# Clean column names by stripping common prefixes used in modeling outputs
prefix = "means_per_cluster_mu_fg_"
if all(isinstance(c, str) and c.startswith(prefix) for c in inf_aver2.columns):
    inf_aver2.columns = [c[len(prefix):] for c in inf_aver2.columns]

# --- Feature and Cell Type Alignment ---
# Intersect and filter genes present in both the spatial data and the reference signature
genes = np.intersect1d(adata_vis.var_names, inf_aver2.index)
adata_vis = adata_vis[:, genes].copy()
inf_aver2 = inf_aver2.loc[genes]

# Align cell types between spatial deconvolution and reference signature
vis_factors = list(adata_vis.uns["mod"]["factor_names"])
common = [ct for ct in vis_factors if ct in inf_aver2.columns]
if not common:
    raise ValueError(f"No overlapping celltypes. vis={vis_factors}, ref={list(inf_aver2.columns)}")

# Ensure cell abundance estimates (from deconvolution) are accessible in the obs table
if not all(ct in adata_vis.obs.columns for ct in common):
    adata_vis.obs[vis_factors] = pd.DataFrame(
        adata_vis.obsm["q05_cell_abundance_w_sf"], index=adata_vis.obs_names, columns=vis_factors
    )

# --- Calculating Expected Spatial Expression ---
# Extract posterior sample means to reconstruct expression based on composition
post = adata_vis.uns["mod"]["post_sample_means"]

# Compute expected expression: Cell Abundance (Spot x CT) @ Reference Signatures (CT x Gene)
total_df = adata_vis.obs[common] @ inf_aver2[common].T

# Apply technical scaling: incorporate gene sensitivity (m_g), additive background (s_g), 
# and spot-level detection efficiency (y_s)
final_df = (total_df.mul(post["m_g"], axis=1) + np.asarray(post["s_g_gene_add"])[0]).mul(post["detection_y_s"], axis=0)

# --- Constructing Comparison Matrices ---
meta_data = adata_vis.obs.copy()
# Combine expression values with spot metadata for both Observed and Expected datasets
observed_expression = pd.concat([adata_vis.to_df(), meta_data], axis=1)
expected_expression = pd.concat([final_df, meta_data], axis=1)

# Assign condition labels and adjust indices to differentiate between datasets
observed_expression["condition"] = "observed"
expected_expression["condition"] = "expected"
observed_expression.index = [f"{i}_obs" for i in observed_expression.index]
expected_expression.index = [f"{i}_exp" for i in expected_expression.index]

# Create a combined AnnData object for downstream statistical testing
combined_df = pd.concat([observed_expression, expected_expression])
drop_cols = list(meta_data.columns) + ["condition"]
adata_tests = sc.AnnData(X=combined_df.drop(columns=drop_cols))
adata_tests.obs["condition"] = combined_df["condition"].values
adata_tests.obs[cluster_info] = combined_df[cluster_info].astype("category")

# --- Final Cleanup and Sanitization ---
# Clean column and cell type names for compatibility with all analysis modules
observed_expression = clean_column_names(observed_expression)
expected_expression = clean_column_names(expected_expression)
meta_data = clean_column_names(meta_data)
inf_aver2 = clean_column_names(inf_aver2)
cell_types = obtain_clean_celltype_names(adata_vis)

print("OK:", observed_expression.shape, expected_expression.shape, "celltypes used:", len(common))

## 3) Detect tumor subtype-specific CCI genes (`ccigenes` module)

To identify genes driven by cell-cell interactions (CCI), we employ a robust statistical framework that integrates two distinct approaches: a `permutation test` to evaluate expression differences across spatial regions and a `chi-square test` to assess variance within each region. The results from both tests are combined using the `Cauchy combination test` to determine overall statistical significance. Final CCI gene candidates are then filtered based on adjusted `p-values` and `log-fold change` (LogFC) thresholds.

Note: This analytical pipeline is consistent across various datasets; only the `SPECIES` and `CLUSTER_INFO` parameters require adjustment to match the specific sample metadata.

In [ ]:
# --- Statistical Testing for CCI Gene Identification ---

# Run a permutation test across all spatial clusters to evaluate 
# significant differences between observed and expected expression levels
perm_results = permutation_test_all_clusters(
    adata_tests,
    cluster_info=cluster_info,
    observed_expression=observed_expression,
    expected_expression=expected_expression,
    n_permutations=N_PERMUTATIONS,
    use_zeros=True,
    random_seed=42,
    path=str(OUTPUT_DIR),
)

# Perform a Chi-square goodness-of-fit test within each cluster 
# to assess the distribution variance between observed and expected data
chi_results = chi_square_goodness_of_fit(
    adata_tests,
    cluster_info=cluster_info,
    groupby="condition",
    reference="observed",
    target="expected",
    use_zeros=True,
)

# --- Statistical Integration ---

# Merge the results from both tests based on gene and spatial cluster identifiers
stats_df = pd.merge(perm_results, chi_results, on=["gene", "cluster"], how="inner")

# Combine the individual p-values into a single meta-p-value using the Cauchy combination test
stats_df["combined_p_value_adj"] = stats_df.apply(compute_combined_p_value, axis=1)

# Save the full statistical report for all tested genes
stats_df.to_csv(OUTPUT_DIR / "allgenes_results.csv", index=False)

# --- Filtering Significant CCI Genes ---

# Apply strict filters: adjusted p-value < 0.01, higher observed mean than expected, 
# and a Log Fold Change (LogFC) greater than the predefined threshold
gene_filter = (
    (stats_df["combined_p_value_adj"] < 0.01)
    & (stats_df["mean_ref"] > stats_df["mean_tgt"])
    & (stats_df["logfc"] > LOG_FC)
)
final_results = stats_df.loc[gene_filter].copy()

# Save the final list of high-confidence CCI (niche) genes
final_results.to_csv(OUTPUT_DIR / "ccigenes_results.csv", index=False)

# --- Results Summary ---

# Display the total number of detected niche genes and a preview of the top candidates
print("Niche gene hits:", final_results.shape[0])
display(final_results[["gene", "cluster", "combined_p_value_adj", "logfc"]].head(20))

## 4) Identify interacting cell-type pairs (`ccipairs` module)

Following the identification of CCI genes, we determine the specific interacting cell-type pairs through a `regression-based framework` (a two-step modeling strategy). In this step, we regress the residual expression signals against cell-type interaction terms to quantify pairwise effects. Finally, these significant interaction terms are biologically annotated by mapping them to curated ligand-receptor databases(`Omnipath`, `CellChat`, `CellTalk`), bridging the gap between observed spatial signals and established molecular signaling pathways.

In [ ]:
# --- Identifying Interacting Cell-Type Pairs (ccipairs) ---

# Check if any significant CCI (niche) genes were identified in the previous step
if final_results.empty:
    print("No niche genes detected; skipping ccipairs.")
else:
    # Normalize cell-type deconvolution data to ensure proportions sum to 1 per spot
    norm_deconv = meta_data.loc[:, cell_types].div(meta_data.loc[:, cell_types].sum(axis=1), axis=0)
    norm_deconv[cluster_info] = meta_data[cluster_info]
    
    # Calculate the average cell-type composition within each spatial cluster (niche)
    cluster_summary = norm_deconv.groupby(cluster_info).mean()
    cluster_summary.loc["mean"] = cluster_summary.mean()

    # Pre-processing: Remove duplicate genes from reference signatures and results to ensure consistency
    inf_aver2 = inf_aver2[~inf_aver2.index.duplicated(keep="first")]
    final_results = final_results.drop_duplicates(subset=["gene"])

    # --- Regression Analysis with Regularization ---
    # Model the residual expression (Observed - Expected) as a function of cell-type interactions.
    # We apply regularization (alpha=0.5) to select the most relevant interacting pairs.
    models_per_gene, gene_analysis = regress_residual_on_interaction_with_regularization(
        observed_expression=observed_expression,
        expected_expression=expected_expression,
        celltypes=cell_types,
        cell_sig=inf_aver2,
        niche_gene_results=final_results,
        cluster_summary=cluster_summary,
        cluster_info=cluster_info,
        self_interaction=False, # Focus on interactions between different cell types
        use_zeros=False,
        alpha=0.5,
    )

    # --- Interaction Term Extraction ---
    # Extract cell-type pairs that significantly contribute to the gene expression residuals
    interaction_terms = extract_interaction_terms(
        models_per_gene, gene_analysis, p_value_threshold=PVAL_TERM
    )

    if interaction_terms.empty:
        print("No significant interaction terms.")
    else:
        # For each gene and cluster, retain only the top 5 most significant interacting pairs
        interaction_terms = (
            interaction_terms.groupby(["cluster", "gene"]) 
            .apply(lambda df: df.sort_values("p_value").head(5))
            .reset_index(drop=True)
        )
        
        # --- Database Comparison and Validation ---
        # Cross-reference identified pairs with curated ligand-receptor databases (e.g., CellChat, CellPhoneDB)
        interaction_terms = compare_database(interaction_terms, species=SPECIES)
        
        # Save the final validated interaction results to a CSV file
        interaction_terms.to_csv(OUTPUT_DIR / "ccipairs_results.csv", index=False)
        
        # Display a preview of the high-confidence interacting cell-type pairs
        display(interaction_terms.head(25))